# SpectralGuard Phase 3 — Automatic H100 Experiments
Select an H100 runtime, then click **Run all** once. If the verified account-1 handoff is uploaded to the repository root, the first cell restores its 2,410 E04 rows automatically. Before mixing old and new evidence, the pipeline proves batch-one/batched equivalence. It then uses exact-length Mamba batches, masked Transformer batches, automatic OOM backoff, atomic resume, a three-hour session timebox with export reserve, and creates `Experiments/exports/phase3_experiments.zip`. Completed examples and attack restarts are never repeated.

In [ ]:
from pathlib import Path
import os
import importlib.util, subprocess, sys
if not Path('requirements-lightning.txt').is_file():
    os.chdir('..')
assert Path('requirements-lightning.txt').is_file(), f'Repository root not found from {Path.cwd()}'
handoff = Path('phase3-handoff-account1.tar.gz')
restored_e04 = Path('Experiments/raw/E04_adaptive.jsonl')
if handoff.is_file() and not restored_e04.is_file():
    subprocess.run([sys.executable, 'scripts/restore_phase3_handoff.py', str(handoff), '--sha256', 'a992faabfd7d9113d25a04ac71f3f1c5fb8424e3ee47d020dc691387e4e3d597'], check=True)
assert sys.version_info[:2] == (3, 12), f'Lightning Phase 3 requires Python 3.12, found {sys.version}'
if importlib.util.find_spec('pip') is None:
    subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'], check=True)
%pip install -q -r requirements-lightning.txt
%pip install -q -e . --no-deps
subprocess.run([sys.executable, 'scripts/install_lightning_mamba_kernels.py', '--cuda-smoke'], check=True)
print('Finished')

In [ ]:
import importlib.metadata as md
import torch
torch_version = md.version('torch').split('+')[0]
assert torch_version == '2.8.0', f'Expected torch 2.8.0, found {md.version("torch")}'
assert str(torch.version.cuda).startswith('12.'), f'Expected a CUDA 12.x torch build, found {torch.version.cuda}'
assert torch.cuda.is_available(), 'STOP: select an H100 runtime before Run all.'
gpu = torch.cuda.get_device_name(0)
assert 'H100' in gpu.upper(), f'STOP: H100 required, detected {gpu}'
assert torch.cuda.is_bf16_supported(), 'STOP: BF16 support is required.'
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu, 'vram_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3})
print('Finished')

In [ ]:
from src.experiments.phase3_runner import run_automatic_phase3
automatic_report = run_automatic_phase3()
assert automatic_report['status'] in {'COMPLETE', 'TIMEBOX_HANDOFF'}
print({key: automatic_report.get(key) for key in ('status', 'session_elapsed_seconds', 'experiments_zip')})
print('Finished')

In [ ]:
from pathlib import Path
complete = automatic_report['status'] == 'COMPLETE'
result = Path('Experiments/exports/phase3_experiments.zip' if complete else 'Experiments/exports/phase3_handoff.tar.gz').resolve()
assert result.is_file() and result.stat().st_size > 0
print(f'DOWNLOAD READY: {result}')
print('Next: stop the H100 and run Notebook 2 on CPU.' if complete else 'Timebox reached safely: stop the H100 and upload this handoff to the next account.')
print('Finished')